# 04 — Recipe control, 1-seed pilot (EXPLORATORY, prereg A7 §c)

**Declared exploratory before running. Never enters `EXPECTED_CELLS`, any Holm family,
or the flip count. Reported whatever it shows.** A7 §d froze the grid at three cells,
so a pass here cannot reinstate the scale arm.

Discriminates the two live explanations for the scale arm's anchor collapse:

| account | prediction for this run |
|---|---|
| augmentation–factor **construct mismatch** | anchor stays low (~0.5) |
| **colour shortcut** left open by a single-augmentation recipe | anchor recovers (>0.9) |

`scale_recipe` is the identical `RandomAffine(scale=(0.75,1.25), fill=0)` composed with
the canonical `ColorJitter` + `RandomGrayscale` base. One seed, ~4 h. Only run the
12-seed version (notebook 05) if this is informative.

**Checkpointing:** `train_simclr` writes `last_ckpt.pt` each epoch, resumes from it
automatically, and skips a run that already finished.

**Setup:** Accelerator `GPU T4 x2`, Internet **On**.

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo
Onto `/kaggle/working` (persists across restarts within a session).

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies
Without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())

## 4. Download shapes3d + build the image cache
`--build-cache` decompresses once into an uncompressed memmap the loaders mmap. Idempotent.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache

## 5. Restore checkpoints from a previous session (resume)
`Add Input -> a prior version's output` (or an uploaded encoder dataset), then run
this. It finds every `scale_recipe_strong_seed*.pt` under `/kaggle/input` and restores it to
`results/encoders/<run_id>/` (`backbone*.pt` -> `backbone.pt`, `last_ckpt*.pt` ->
`last_ckpt.pt`). On a fresh first run there is nothing to restore. Only scale_recipe
files are touched.

In [ ]:
import re, shutil
from pathlib import Path

REPO  = Path("/kaggle/working/probe-capacity-invariance")
ENC   = REPO / "results" / "encoders"
INPUT = Path("/kaggle/input")
_RID  = re.compile(r"scale_recipe_strong_seed\d+")

def _target(name):
    n = name.lower()
    if "ckpt" in n:     return "last_ckpt.pt"
    if "backbone" in n: return "backbone.pt"
    return None

found = {}
for p in sorted(INPUT.rglob("*.pt")) if INPUT.exists() else []:
    m, tgt = _RID.search(p.as_posix()), _target(p.name)
    if m and tgt:
        found.setdefault((m.group(0), tgt), p)     # first match per (run_id, kind)

if not found:
    print("nothing to restore -- fresh start")
for (rid, tgt), src in sorted(found.items()):
    dst = ENC / rid / tgt
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        shutil.copy2(src, dst)
    print(f"{rid:34s} {tgt:14s} <- {src}")

## 6. Integrity check — keep good checkpoints, purge corrupt ones
Each restored `.pt` is opened as a zip. A valid `backbone.pt` means the seed is
done and training skips it. A corrupt `backbone.pt` is deleted with its
`last_ckpt.pt` so the seed retrains; a valid `last_ckpt.pt` with no backbone lets
training resume mid-run.

In [ ]:
import zipfile
from pathlib import Path

ENC = Path("/kaggle/working/probe-capacity-invariance/results/encoders")

def _state(p):
    if not p.exists():
        return "missing"
    try:
        return None if zipfile.ZipFile(p).testzip() is None else "corrupt"
    except Exception as e:
        return f"not-a-zip ({e})"

for d in sorted(ENC.glob("scale_recipe_strong_seed*")):
    bb, ck = d / "backbone.pt", d / "last_ckpt.pt"
    bstat = _state(bb)
    if bstat is None:
        print(f"{d.name:34s} backbone OK -> skip"); continue
    if bstat != "missing":
        bb.unlink(missing_ok=True); ck.unlink(missing_ok=True)
        print(f"{d.name:34s} backbone {bstat} -> purged, will retrain"); continue
    print(f"{d.name:34s} "
          + ("last_ckpt OK -> resume" if _state(ck) is None else "fresh start"))

## 7. Train seed 0 (~4 h)
If the session times out: Save Version, Add Input -> this output, re-run. Section 5 restores `last_ckpt.pt` and training resumes.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.encoders.train_simclr \
    --config configs/run/scale_recipe_control.yaml --set run.seed=0

## 8. Gate check — shape anchor across the ladder vs the random floor

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.run_sweep \
    --config configs/probe/ladder.yaml \
    --dataset shapes3d --condition scale_recipe --strength strong \
    --encoders results/encoders/scale_recipe_strong_seed*/backbone.pt \
    --random-seed 0 1 2 --subsample 40000 \
    --device cuda --num-workers 2 --resume --out-root results/probes_excluded

## 9. Verdict

In [ ]:
import numpy as np, json
from pathlib import Path
d = Path("/kaggle/working/probe-capacity-invariance") / "results/probes_excluded/scale_recipe_strong"
z = np.load(d / "stacks.npz"); m = json.loads((d / "meta.json").read_text())
si = [f["name"] for f in m["factors"]].index("shape")
tr, rn = z["trained"][:, si, :].mean(0), z["random"][:, si, :].mean(0)
print("rung          trained    floor")
for i, r in enumerate(m["rungs"]):
    print(f"{r:12s}{tr[i]:>9.4f}{rn[i]:>9.4f}")
print("\nbaseline, scale_strong without the photometric base: 0.483 / 0.536 / 0.564 / 0.609")
print("VERDICT:",
      "colour-shortcut account supported -- the recipe caused the collapse" if tr[0] >= 0.90
      else "construct-mismatch account survives -- the transform destroys the anchor"
      if tr[0] < 0.65 else "ambiguous; report as such")

In [ ]:
# --- persist for the next session --------------------------------------------
# /kaggle/working is the notebook's output. Click "Save Version" when this
# finishes, then Add Input -> this output on the next run to resume.
import shutil
from pathlib import Path
src = Path("/kaggle/working/probe-capacity-invariance/results"); dst = Path("/kaggle/working/results")
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
print(f"persisted {sum(1 for _ in dst.rglob('*') if _.is_file())} files "
      f"-> click 'Save Version' now")